In [4]:
import nibabel as nib
import numpy as np
from scipy.ndimage import median_filter
from skimage import exposure
from skimage.restoration import denoise_bilateral
from scipy.ndimage import zoom
import tomllib


with open("../src/config.toml", mode="rb") as f:
    config = tomllib.load(f)

sequence = "T1"

if "T2":
    base            = config["data"]["base"]
    input_file      = base + config["data"]["nii_file_T2"]
else:
    base            = config["data"]["base"]
    input_file      = base + config["data"]["nii_file"]


output_file = input_file[:-4] + "_cropped_denoised.nii"

In [ ]:
def crop_nifti(input_path, output_path, zoom_factor, x_range, y_range, z_range, kernel_size):
    """
    Crop a 3D NIfTI image and save it with updated affine.

    Parameters:
        input_path (str): input .nii/.nii.gz
        output_path (str): output file
        x_range, y_range, z_range: tuples (start, end)
    """

    nii = nib.load(input_path)
    data = np.asarray(nii.dataobj)  # efficient loading
    affine = nii.affine.copy()
    header = nii.header.copy()


    pixel_spacing = header.get_zooms()[:3] 
    spacing_file = input_path[:-4] + "_pixel_spacing.txt"
    
    with open(spacing_file, 'w') as f:
        # Writes it as space-separated values, or customize as you prefer
        f.write(f"{pixel_spacing[0]} {pixel_spacing[1]} {pixel_spacing[2]}\n")
    print(f"Original pixel spacing saved to: {spacing_file} -> {pixel_spacing}")


    print(data.shape)

    # Crop data
    cropped = data[
        x_range[0]:x_range[1],
        y_range[0]:y_range[1],
        z_range[0]:z_range[1]
    ]


    # --- Normalize (important before CLAHE) ---
    c_min, c_max = cropped.min(), cropped.max()
    norm = (cropped - c_min) / (c_max - c_min + 1e-8)

    # --- CLAHE (slice-by-slice) ---
    equalized = np.zeros_like(norm)

    for z in range(norm.shape[2]):
        equalized[:, :, z] = exposure.equalize_adapthist(
            norm[:, :, z],
            clip_limit=0.015  # tweak this
        )

        equalized[:, :, z] = denoise_bilateral(
            equalized[:, :, z],
            sigma_color=0.1,
            sigma_spatial=3
        )

    # --- Rescale back to original intensity range ---
    equalized = equalized * (c_max - c_min) + c_min

    # Optional: restore dtype
    equalized = equalized.astype(nii.get_data_dtype())

    equalized = median_filter(equalized, size=kernel_size)


    # --- Update affine ---
    # Compute offset in voxel space
    offset = np.array([x_range[0], y_range[0], z_range[0], 1])

    # Convert voxel offset to world coordinates
    new_origin = affine @ offset

    # Update translation part of affine
    new_affine = affine.copy()
    new_affine[:3, 3] = new_origin[:3]

    # np.savetxt(output_file[:-4] + "_modified_offset.txt", new_origin[:3])


    if zoom_factor > 1:
        # --- Resample ---
        zoomed = zoom(equalized, zoom=zoom_factor, order=3)  # cubic interpolation

        # --- Update affine ---
        new_affine[:3, :3] /= zoom_factor  # voxel size becomes smaller
        equalized = zoomed




    # Create new NIfTI
    cropped_nii = nib.Nifti1Image(equalized, new_affine, header)

    # Save
    nib.save(cropped_nii, output_path)

    print(f"Cropped and denoised volume saved to: {output_path}")


crop_nifti(input_file, output_file, 1, [120, 260], [40,220], [10,90], 2)


Original pixel spacing saved to: /Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/B_FFE i0.9 XD Fast Gch/1601_pixel_spacing.txt -> (np.float32(0.75), np.float32(0.75), np.float32(0.7500004))
(320, 320, 100)
Cropped and denoised volume saved to: /Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/B_FFE i0.9 XD Fast Gch/1601_cropped_denoised.nii
